Create comprehensive Python code implementation for the bio-inspired optimization algorithms (GA, PSO, ACO, FA, BFO) for FACTS device placement in power systems. The code should include:

Complete implementation of all 5 metaheuristic algorithms (GA, PSO, ACO, FA, BFO)
AI-enhanced versions (NEGA, DRL-PSO, ANFFA) using neural networks and reinforcement learning
FACTS device models (SVC, TCSC, UPFC, DPFC) with mathematical formulations
Power flow calculations and OPF problem formulation
Multi-objective optimization with Pareto fronts
IEEE test system data and renewable energy integration models
Performance evaluation and comparison functions
Visualization tools for results and convergence plots
Complete documentation and comments explaining each component
Save all code files as separate Python modules with proper organization and structure. Include a main execution script that demonstrates the full workflow.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import minimize
import networkx as nx
from collections import defaultdict
import pickle
import json
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Create output directory
output_dir = '/home/user/output'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

print(f"Output directory created: {output_dir}")
print("Starting comprehensive bio-inspired optimization implementation...")

In [ ]:
# 1. Base Classes and Utilities
base_classes_code = '''
"""
Base Classes and Utilities for Bio-Inspired Optimization for FACTS Placement 31
Author: AI-Enhanced Power Systems Research Group
Date: 2024
"""

import numpy as np
import pandas as pd
from abc import ABC, abstractmethod
from typing import List, Dict, Tuple, Optional, Union
import logging
from dataclasses import dataclass
from enum import Enum

# Setup logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

class FACTSType(Enum):
    """Enumeration for FACTS device types"""
    SVC = "Static VAR Compensator"
    TCSC = "Thyristor Controlled Series Capacitor"
    UPFC = "Unified Power Flow Controller"
    DPFC = "Distributed Power Flow Controller"

@dataclass
class SystemParameters:
    """System parameters and constraints"""
    base_mva: float = 100.0
    frequency: float = 60.0
    voltage_tolerance: float = 0.05
    max_iterations: int = 100
    convergence_tolerance: float = 1e-6

@dataclass
class Bus:
    """Bus data structure"""
    number: int
    voltage_mag: float
    voltage_angle: float
    load_p: float
    load_q: float
    gen_p: float
    gen_q: float
    bus_type: int  # 1: PQ, 2: PV, 3: Slack
    v_min: float = 0.95
    v_max: float = 1.05

@dataclass
class Branch:
    """Transmission line data structure"""
    from_bus: int
    to_bus: int
    resistance: float
    reactance: float
    susceptance: float
    rating: float
    tap_ratio: float = 1.0

@dataclass
class Generator:
    """Generator data structure"""
    bus: int
    p_gen: float
    q_gen: float
    p_min: float
    p_max: float
    q_min: float
    q_max: float
    cost_a: float  # Cost coefficient a
    cost_b: float  # Cost coefficient b
    cost_c: float  # Cost coefficient c

@dataclass
class FACTSDevice:
    """FACTS device data structure"""
    device_type: FACTSType
    location: Union[int, Tuple[int, int]]  # Bus number for SVC, line tuple for others
    rating: float
    parameters: Dict[str, float]
    status: bool = True

class ObjectiveFunction:
    """Base class for objective functions"""
    
    @staticmethod
    def generation_cost(generators: List[Generator]) -> float:
        """Calculate total generation cost"""
        total_cost = 0.0
        for gen in generators:
            total_cost += gen.cost_a + gen.cost_b * gen.p_gen + gen.cost_c * (gen.p_gen ** 2)
        return total_cost
    
    @staticmethod
    def transmission_losses(buses: List[Bus], branches: List[Branch]) -> float:
        """Calculate total transmission losses"""
        total_losses = 0.0
        for branch in branches:
            from_bus = next(b for b in buses if b.number == branch.from_bus)
            to_bus = next(b for b in buses if b.number == branch.to_bus)
            
            # Calculate line losses using power flow equations
            g = branch.resistance / (branch.resistance**2 + branch.reactance**2)
            losses = g * (from_bus.voltage_mag**2 + to_bus.voltage_mag**2 - 
                         2*from_bus.voltage_mag*to_bus.voltage_mag*
                         np.cos(from_bus.voltage_angle - to_bus.voltage_angle))
            total_losses += losses
        
        return total_losses
    
    @staticmethod
    def voltage_deviation(buses: List[Bus], reference_voltage: float = 1.0) -> float:
        """Calculate total voltage deviation"""
        total_deviation = 0.0
        for bus in buses:
            total_deviation += (bus.voltage_mag - reference_voltage)**2
        return total_deviation
    
    @staticmethod
    def stability_index(buses: List[Bus], branches: List[Branch]) -> float:
        """Calculate system stability index"""
        stability_sum = 0.0
        for branch in branches:
            from_bus = next(b for b in buses if b.number == branch.from_bus)
            to_bus = next(b for b in buses if b.number == branch.to_bus)
            
            # Simplified stability calculation
            s_ij = abs(from_bus.voltage_mag * to_bus.voltage_mag * 
                      np.sin(from_bus.voltage_angle - to_bus.voltage_angle) / branch.reactance)
            stability_sum += s_ij**2
        
        return stability_sum

class PowerSystemNetwork:
    """Power system network representation"""
    
    def __init__(self):
        self.buses: List[Bus] = []
        self.branches: List[Branch] = []
        self.generators: List[Generator] = []
        self.facts_devices: List[FACTSDevice] = []
        self.system_params = SystemParameters()
        
    def add_bus(self, bus: Bus):
        """Add bus to the network"""
        self.buses.append(bus)
        
    def add_branch(self, branch: Branch):
        """Add branch to the network"""
        self.branches.append(branch)
        
    def add_generator(self, generator: Generator):
        """Add generator to the network"""
        self.generators.append(generator)
        
    def add_facts_device(self, device: FACTSDevice):
        """Add FACTS device to the network"""
        self.facts_devices.append(device)
        
    def get_admittance_matrix(self) -> np.ndarray:
        """Calculate system admittance matrix"""
        n_buses = len(self.buses)
        Y = np.zeros((n_buses, n_buses), dtype=complex)
        
        # Add branch admittances
        for branch in self.branches:
            i = branch.from_bus - 1  # Convert to 0-based indexing
            j = branch.to_bus - 1
            
            # Calculate admittance
            z = complex(branch.resistance, branch.reactance)
            y = 1.0 / z
            
            # Add to Y matrix
            Y[i, i] += y + complex(0, branch.susceptance/2)
            Y[j, j] += y + complex(0, branch.susceptance/2)
            Y[i, j] -= y
            Y[j, i] -= y
            
        return Y
    
    def power_flow_newton_raphson(self, max_iter: int = 50, tolerance: float = 1e-6) -> bool:
        """Solve power flow using Newton-Raphson method"""
        n_buses = len(self.buses)
        Y = self.get_admittance_matrix()
        
        # Initialize voltage vectors
        V = np.array([complex(bus.voltage_mag * np.cos(bus.voltage_angle),
                             bus.voltage_mag * np.sin(bus.voltage_angle)) 
                     for bus in self.buses])
        
        for iteration in range(max_iter):
            # Calculate power mismatches
            P_calc = np.zeros(n_buses)
            Q_calc = np.zeros(n_buses)
            
            for i in range(n_buses):
                for j in range(n_buses):
                    P_calc[i] += abs(V[i]) * abs(V[j]) * (
                        np.real(Y[i, j]) * np.cos(np.angle(V[i]) - np.angle(V[j])) +
                        np.imag(Y[i, j]) * np.sin(np.angle(V[i]) - np.angle(V[j]))
                    )
                    Q_calc[i] += abs(V[i]) * abs(V[j]) * (
                        np.imag(Y[i, j]) * np.cos(np.angle(V[i]) - np.angle(V[j])) -
                        np.real(Y[i, j]) * np.sin(np.angle(V[i]) - np.angle(V[j]))
                    )
            
            # Calculate mismatches
            P_spec = np.array([bus.gen_p - bus.load_p for bus in self.buses])
            Q_spec = np.array([bus.gen_q - bus.load_q for bus in self.buses])
            
            dP = P_spec - P_calc
            dQ = Q_spec - Q_calc
            
            # Check convergence
            if np.max(np.abs(np.concatenate([dP[1:], dQ[1:]]))) < tolerance:
                # Update bus voltages
                for i, bus in enumerate(self.buses):
                    bus.voltage_mag = abs(V[i])
                    bus.voltage_angle = np.angle(V[i])
                return True
            
            # Build Jacobian matrix (simplified)
            # In practice, this would be more detailed
            J = np.eye(2*(n_buses-1))  # Simplified placeholder
            
            # Solve for corrections (simplified)
            dx = np.linalg.solve(J, np.concatenate([dP[1:], dQ[1:]]))
            
            # Update voltages (simplified)
            for i in range(1, n_buses):
                V[i] *= (1 + 0.01 * dx[i-1])  # Simplified update
        
        logger.warning("Power flow did not converge")
        return False

class OptimizationProblem:
    """Multi-objective optimization problem formulation"""
    
    def __init__(self, network: PowerSystemNetwork):
        self.network = network
        self.objectives = []
        self.constraints = []
        
    def add_objective(self, objective_func, weight: float = 1.0):
        """Add objective function"""
        self.objectives.append((objective_func, weight))
        
    def add_constraint(self, constraint_func):
        """Add constraint function"""
        self.constraints.append(constraint_func)
        
    def evaluate(self, solution: np.ndarray) -> Tuple[List[float], bool]:
        """Evaluate solution"""
        # Apply solution to network
        self.apply_solution_to_network(solution)
        
        # Solve power flow
        if not self.network.power_flow_newton_raphson():
            return [float('inf')] * len(self.objectives), False
        
        # Evaluate objectives
        objective_values = []
        for obj_func, weight in self.objectives:
            if obj_func.__name__ == 'generation_cost':
                value = ObjectiveFunction.generation_cost(self.network.generators)
            elif obj_func.__name__ == 'transmission_losses':
                value = ObjectiveFunction.transmission_losses(self.network.buses, self.network.branches)
            elif obj_func.__name__ == 'voltage_deviation':
                value = ObjectiveFunction.voltage_deviation(self.network.buses)
            elif obj_func.__name__ == 'stability_index':
                value = ObjectiveFunction.stability_index(self.network.buses, self.network.branches)
            else:
                value = obj_func(self.network)
            
            objective_values.append(weight * value)
        
        # Check constraints
        feasible = self.check_constraints()
        
        return objective_values, feasible
    
    def apply_solution_to_network(self, solution: np.ndarray):
        """Apply optimization solution to network"""
        # This method would decode the solution vector and apply
        # generator settings, FACTS device parameters, etc.
        pass
    
    def check_constraints(self) -> bool:
        """Check if current solution satisfies constraints"""
        # Check voltage limits
        for bus in self.network.buses:
            if bus.voltage_mag < bus.v_min or bus.voltage_mag > bus.v_max:
                return False
        
        # Check generator limits
        for gen in self.network.generators:
            if gen.p_gen < gen.p_min or gen.p_gen > gen.p_max:
                return False
            if gen.q_gen < gen.q_min or gen.q_gen > gen.q_max:
                return False
        
        return True

# Utility functions
def create_ieee_test_system(system_name: str) -> PowerSystemNetwork:
    """Create IEEE test system"""
    network = PowerSystemNetwork()
    
    if system_name == "IEEE9":
        # IEEE 9-bus system data
        buses_data = [
            (1, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 3),  # Slack bus
            (2, 1.0, 0.0, 0.0, 0.0, 163.0, 0.0, 2),  # PV bus
            (3, 1.0, 0.0, 0.0, 0.0, 85.0, 0.0, 2),   # PV bus
            (4, 1.0, 0.0, 125.0, 50.0, 0.0, 0.0, 1), # Load bus
            (5, 1.0, 0.0, 90.0, 30.0, 0.0, 0.0, 1),  # Load bus
            (6, 1.0, 0.0, 100.0, 35.0, 0.0, 0.0, 1), # Load bus
            (7, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1),    # Junction
            (8, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1),    # Junction
            (9, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1)     # Junction
        ]
        
        for bus_data in buses_data:
            bus = Bus(*bus_data)
            network.add_bus(bus)
            
        # Add generators
        generators_data = [
            (1, 0.0, 0.0, 0.0, 250.0, -300.0, 300.0, 1500.0, 10.0, 0.05),
            (2, 163.0, 0.0, 10.0, 300.0, -300.0, 300.0, 2000.0, 15.0, 0.06),
            (3, 85.0, 0.0, 10.0, 270.0, -300.0, 300.0, 3000.0, 25.0, 0.08)
        ]
        
        for gen_data in generators_data:
            gen = Generator(*gen_data)
            network.add_generator(gen)
    
    # Add more test systems as needed...
    
    return network

def pareto_dominance(obj1: List[float], obj2: List[float]) -> int:
    """
    Check Pareto dominance relationship
    Returns: 1 if obj1 dominates obj2, -1 if obj2 dominates obj1, 0 if non-dominated
    """
    better = False
    worse = False
    
    for i in range(len(obj1)):
        if obj1[i] < obj2[i]:
            better = True
        elif obj1[i] > obj2[i]:
            worse = True
    
    if better and not worse:
        return 1
    elif worse and not better:
        return -1
    else:
        return 0

def calculate_hypervolume(pareto_front: List[List[float]], reference_point: List[float]) -> float:
    """Calculate hypervolume indicator"""
    # Simplified hypervolume calculation
    if not pareto_front:
        return 0.0
    
    volume = 0.0
    for solution in pareto_front:
        prod = 1.0
        for i in range(len(solution)):
            prod *= max(0, reference_point[i] - solution[i])
        volume += prod
    
    return volume
'''

# Save base classes
with open(f'{output_dir}/base_classes.py', 'w') as f:
    f.write(base_classes_code)

print("✓ Base classes and utilities created")

In [ ]:
# 2. FACTS Device Models
facts_models_code = '''
"""
FACTS Device Models for Bio-Inspired Optimization for FACTS Placement 31
Author: AI-Enhanced Power Systems Research Group
Date: 2024

This module contains mathematical models for various FACTS devices:
- SVC (Static VAR Compensator)
- TCSC (Thyristor Controlled Series Capacitor)  
- UPFC (Unified Power Flow Controller)
- DPFC (Distributed Power Flow Controller)
"""

import numpy as np
from typing import Dict, Tuple, List
from dataclasses import dataclass
from base_classes import FACTSType, FACTSDevice, PowerSystemNetwork, Bus, Branch
import logging

logger = logging.getLogger(__name__)

class SVCModel:
    """Static VAR Compensator Model"""
    
    def __init__(self, bus_number: int, susceptance_range: Tuple[float, float]):
        self.bus_number = bus_number
        self.b_min, self.b_max = susceptance_range
        self.susceptance = 0.0  # Current susceptance value
        
    def calculate_reactive_power(self, voltage: float) -> float:
        """Calculate reactive power injection"""
        return -voltage**2 * self.susceptance
    
    def update_admittance_matrix(self, Y_matrix: np.ndarray) -> np.ndarray:
        """Update system admittance matrix with SVC"""
        bus_idx = self.bus_number - 1  # Convert to 0-based indexing
        Y_matrix[bus_idx, bus_idx] += complex(0, self.susceptance)
        return Y_matrix
    
    def set_susceptance(self, susceptance: float) -> bool:
        """Set SVC susceptance within limits"""
        if self.b_min <= susceptance <= self.b_max:
            self.susceptance = susceptance
            return True
        return False
    
    def get_parameters(self) -> Dict[str, float]:
        """Get SVC parameters"""
        return {
            'susceptance': self.susceptance,
            'b_min': self.b_min,
            'b_max': self.b_max,
            'reactive_power_rating': abs(self.b_max - self.b_min) * 1.0**2  # Assuming 1 p.u. voltage
        }

class TCSCModel:
    """Thyristor Controlled Series Capacitor Model"""
    
    def __init__(self, from_bus: int, to_bus: int, reactance_range: Tuple[float, float]):
        self.from_bus = from_bus
        self.to_bus = to_bus
        self.x_min, self.x_max = reactance_range
        self.reactance = 0.0  # Current reactance value
        
    def calculate_impedance(self) -> complex:
        """Calculate TCSC impedance"""
        return complex(0, self.reactance)
    
    def update_admittance_matrix(self, Y_matrix: np.ndarray, line_reactance: float) -> np.ndarray:
        """Update system admittance matrix with TCSC"""
        from_idx = self.from_bus - 1
        to_idx = self.to_bus - 1
        
        # Remove original line admittance
        y_line = 1.0 / complex(0, line_reactance)
        Y_matrix[from_idx, from_idx] -= y_line
        Y_matrix[to_idx, to_idx] -= y_line
        Y_matrix[from_idx, to_idx] += y_line
        Y_matrix[to_idx, from_idx] += y_line
        
        # Add TCSC modified admittance
        total_reactance = line_reactance + self.reactance
        y_new = 1.0 / complex(0, total_reactance)
        Y_matrix[from_idx, from_idx] += y_new
        Y_matrix[to_idx, to_idx] += y_new
        Y_matrix[from_idx, to_idx] -= y_new
        Y_matrix[to_idx, from_idx] -= y_new
        
        return Y_matrix
    
    def set_reactance(self, reactance: float) -> bool:
        """Set TCSC reactance within limits"""
        if self.x_min <= reactance <= self.x_max:
            self.reactance = reactance
            return True
        return False
    
    def get_parameters(self) -> Dict[str, float]:
        """Get TCSC parameters"""
        return {
            'reactance': self.reactance,
            'x_min': self.x_min,
            'x_max': self.x_max,
            'compensation_degree': self.reactance / (self.x_max - self.x_min) if (self.x_max != self.x_min) else 0.0
        }

class UPFCModel:
    """Unified Power Flow Controller Model"""
    
    def __init__(self, from_bus: int, to_bus: int, rating: float):
        self.from_bus = from_bus
        self.to_bus = to_bus
        self.rating = rating  # MVA rating
        
        # Control variables
        self.v_series_mag = 0.0  # Series voltage magnitude (p.u.)
        self.v_series_angle = 0.0  # Series voltage angle (rad)
        self.v_shunt_mag = 1.0  # Shunt converter voltage magnitude (p.u.)
        self.v_shunt_angle = 0.0  # Shunt converter voltage angle (rad)
        
        # Limits
        self.v_series_max = 0.1  # Maximum series voltage (10% of line voltage)
        self.angle_max = np.pi/4  # Maximum angle control range
        
    def calculate_series_voltage(self) -> complex:
        """Calculate series injected voltage"""
        return self.v_series_mag * np.exp(1j * self.v_series_angle)
    
    def calculate_shunt_voltage(self) -> complex:
        """Calculate shunt converter voltage"""
        return self.v_shunt_mag * np.exp(1j * self.v_shunt_angle)
    
    def update_power_equations(self, V_from: complex, V_to: complex, Y_line: complex) -> Tuple[complex, complex]:
        """Update power flow equations with UPFC"""
        V_series = self.calculate_series_voltage()
        
        # Modified voltages
        V_from_modified = V_from
        V_to_modified = V_to - V_series
        
        # Power flows
        S_from = V_from_modified * np.conj((V_from_modified - V_to_modified) * Y_line)
        S_to = V_to_modified * np.conj((V_to_modified - V_from_modified) * Y_line)
        
        return S_from, S_to
    
    def set_control_parameters(self, v_mag: float, v_angle: float, shunt_mag: float = None, shunt_angle: float = None) -> bool:
        """Set UPFC control parameters"""
        if 0 <= v_mag <= self.v_series_max and -self.angle_max <= v_angle <= self.angle_max:
            self.v_series_mag = v_mag
            self.v_series_angle = v_angle
            
            if shunt_mag is not None:
                self.v_shunt_mag = max(0.9, min(1.1, shunt_mag))
            if shunt_angle is not None:
                self.v_shunt_angle = max(-self.angle_max, min(self.angle_max, shunt_angle))
            
            return True
        return False
    
    def get_parameters(self) -> Dict[str, float]:
        """Get UPFC parameters"""
        return {
            'v_series_mag': self.v_series_mag,
            'v_series_angle': self.v_series_angle,
            'v_shunt_mag': self.v_shunt_mag,
            'v_shunt_angle': self.v_shunt_angle,
            'rating': self.rating,
            'series_utilization': self.v_series_mag / self.v_series_max
        }

class DPFCModel:
    """Distributed Power Flow Controller Model"""
    
    def __init__(self, lines: List[Tuple[int, int]], rating: float):
        self.lines = lines  # List of (from_bus, to_bus) tuples
        self.rating = rating
        
        # Series converters for each line
        self.series_converters = {}
        for line in lines:
            self.series_converters[line] = {
                'v_mag': 0.0,
                'v_angle': 0.0
            }
        
        # Common DC link
        self.dc_voltage = 1.0  # p.u.
        
        # Limits
        self.v_series_max = 0.08  # Maximum series voltage (8% of line voltage)
        self.angle_max = np.pi/6  # Maximum angle control range
        
    def calculate_series_voltage(self, line: Tuple[int, int]) -> complex:
        """Calculate series voltage for specific line"""
        if line in self.series_converters:
            converter = self.series_converters[line]
            return converter['v_mag'] * np.exp(1j * converter['v_angle'])
        return 0.0 + 0.0j
    
    def update_line_power_flow(self, line: Tuple[int, int], V_from: complex, V_to: complex, Y_line: complex) -> Tuple[complex, complex]:
        """Update power flow for specific line with DPFC"""
        V_series = self.calculate_series_voltage(line)
        
        # Modified voltage at receiving end
        V_to_modified = V_to - V_series
        
        # Power flows
        S_from = V_from * np.conj((V_from - V_to_modified) * Y_line)
        S_to = V_to_modified * np.conj((V_to_modified - V_from) * Y_line)
        
        return S_from, S_to
    
    def set_converter_parameters(self, line: Tuple[int, int], v_mag: float, v_angle: float) -> bool:
        """Set parameters for specific series converter"""
        if line in self.series_converters:
            if 0 <= v_mag <= self.v_series_max and -self.angle_max <= v_angle <= self.angle_max:
                self.series_converters[line]['v_mag'] = v_mag
                self.series_converters[line]['v_angle'] = v_angle
                return True
        return False
    
    def check_dc_link_constraint(self) -> bool:
        """Check DC link power balance constraint"""
        total_active_power = 0.0
        for line, converter in self.series_converters.items():
            # Simplified calculation - in practice would be more complex
            total_active_power += converter['v_mag'] * np.cos(converter['v_angle'])
        
        return abs(total_active_power) < 0.01  # Small tolerance for numerical errors
    
    def get_parameters(self) -> Dict[str, Dict[str, float]]:
        """Get DPFC parameters"""
        params = {
            'dc_voltage': self.dc_voltage,
            'rating': self.rating,
            'converters': {}
        }
        
        for line, converter in self.series_converters.items():
            params['converters'][f"line_{line[0]}_{line[1]}"] = {
                'v_mag': converter['v_mag'],
                'v_angle': converter['v_angle'],
                'utilization': converter['v_mag'] / self.v_series_max
            }
        
        return params

class FACTSDeviceFactory:
    """Factory class for creating FACTS devices"""
    
    @staticmethod
    def create_svc(bus_number: int, rating: float) -> SVCModel:
        """Create SVC device"""
        # Convert MVAr rating to susceptance range
        b_max = rating / (1.0**2)  # Assuming 1 p.u. voltage
        b_min = -b_max
        return SVCModel(bus_number, (b_min, b_max))
    
    @staticmethod
    def create_tcsc(from_bus: int, to_bus: int, compensation_range: Tuple[float, float]) -> TCSCModel:
        """Create TCSC device"""
        return TCSCModel(from_bus, to_bus, compensation_range)
    
    @staticmethod
    def create_upfc(from_bus: int, to_bus: int, rating: float) -> UPFCModel:
        """Create UPFC device"""
        return UPFCModel(from_bus, to_bus, rating)
    
    @staticmethod
    def create_dpfc(lines: List[Tuple[int, int]], rating: float) -> DPFCModel:
        """Create DPFC device"""
        return DPFCModel(lines, rating)

class FACTSIntegratedPowerFlow:
    """Power flow solver with FACTS devices integration"""
    
    def __init__(self, network: PowerSystemNetwork):
        self.network = network
        self.facts_models = {}
        self._initialize_facts_models()
    
    def _initialize_facts_models(self):
        """Initialize FACTS device models from network"""
        for device in self.network.facts_devices:
            if device.device_type == FACTSType.SVC:
                self.facts_models[id(device)] = FACTSDeviceFactory.create_svc(
                    device.location, device.rating
                )
            elif device.device_type == FACTSType.TCSC and isinstance(device.location, tuple):
                # Assume compensation range from parameters or default
                comp_range = device.parameters.get('reactance_range', (-0.7, 0.2))
                self.facts_models[id(device)] = FACTSDeviceFactory.create_tcsc(
                    device.location[0], device.location[1], comp_range
                )
            elif device.device_type == FACTSType.UPFC and isinstance(device.location, tuple):
                self.facts_models[id(device)] = FACTSDeviceFactory.create_upfc(
                    device.location[0], device.location[1], device.rating
                )
            elif device.device_type == FACTSType.DPFC:
                # Extract lines from parameters
                lines = device.parameters.get('lines', [])
                self.facts_models[id(device)] = FACTSDeviceFactory.create_dpfc(
                    lines, device.rating
                )
    
    def solve_power_flow_with_facts(self, max_iter: int = 50, tolerance: float = 1e-6) -> bool:
        """Solve power flow with FACTS devices"""
        n_buses = len(self.network.buses)
        
        for iteration in range(max_iter):
            # Get base admittance matrix
            Y = self.network.get_admittance_matrix()
            
            # Modify admittance matrix for FACTS devices
            for device_id, facts_model in self.facts_models.items():
                if isinstance(facts_model, SVCModel):
                    Y = facts_model.update_admittance_matrix(Y)
                elif isinstance(facts_model, TCSCModel):
                    # Find corresponding line reactance
                    line_x = self._get_line_reactance(facts_model.from_bus, facts_model.to_bus)
                    Y = facts_model.update_admittance_matrix(Y, line_x)
            
            # Solve power flow (simplified Newton-Raphson)
            if self._newton_raphson_iteration(Y, tolerance):
                logger.info(f"Power flow converged in {iteration + 1} iterations")
                return True
        
        logger.warning("Power flow with FACTS did not converge")
        return False
    
    def _get_line_reactance(self, from_bus: int, to_bus: int) -> float:
        """Get line reactance between two buses"""
        for branch in self.network.branches:
            if (branch.from_bus == from_bus and branch.to_bus == to_bus) or \\
               (branch.from_bus == to_bus and branch.to_bus == from_bus):
                return branch.reactance
        return 0.1  # Default value if not found
    
    def _newton_raphson_iteration(self, Y: np.ndarray, tolerance: float) -> bool:
        """Single Newton-Raphson iteration (simplified)"""
        # This is a simplified implementation
        # In practice, this would include full Jacobian calculation and update
        return True  # Placeholder for convergence
    
    def update_facts_parameters(self, device_id: int, parameters: Dict[str, float]):
        """Update FACTS device parameters"""
        if device_id in self.facts_models:
            facts_model = self.facts_models[device_id]
            
            if isinstance(facts_model, SVCModel):
                if 'susceptance' in parameters:
                    facts_model.set_susceptance(parameters['susceptance'])
            
            elif isinstance(facts_model, TCSCModel):
                if 'reactance' in parameters:
                    facts_model.set_reactance(parameters['reactance'])
            
            elif isinstance(facts_model, UPFCModel):
                v_mag = parameters.get('v_series_mag', facts_model.v_series_mag)
                v_angle = parameters.get('v_series_angle', facts_model.v_series_angle)
                facts_model.set_control_parameters(v_mag, v_angle)
            
            elif isinstance(facts_model, DPFCModel):
                for line_key, line_params in parameters.items():
                    if line_key.startswith('line_'):
                        # Parse line identifier
                        parts = line_key.split('_')
                        if len(parts) >= 3:
                            from_bus, to_bus = int(parts[1]), int(parts[2])
                            v_mag = line_params.get('v_mag', 0.0)
                            v_angle = line_params.get('v_angle', 0.0)
                            facts_model.set_converter_parameters((from_bus, to_bus), v_mag, v_angle)
    
    def get_all_facts_parameters(self) -> Dict[int, Dict[str, float]]:
        """Get parameters of all FACTS devices"""
        all_params = {}
        for device_id, facts_model in self.facts_models.items():
            all_params[device_id] = facts_model.get_parameters()
        return all_params

# Utility functions for FACTS placement optimization
def evaluate_facts_placement_benefit(network: PowerSystemNetwork, 
                                   device_type: FACTSType, 
                                   location: Union[int, Tuple[int, int]], 
                                   rating: float) -> Dict[str, float]:
    """Evaluate the benefit of placing a FACTS device at a specific location"""
    
    # Create a copy of the network for testing
    test_network = PowerSystemNetwork()
    test_network.buses = network.buses.copy()
    test_network.branches = network.branches.copy()
    test_network.generators = network.generators.copy()
    
    # Calculate baseline performance
    baseline_solver = FACTSIntegratedPowerFlow(test_network)
    baseline_solver.solve_power_flow_with_facts()
    
    baseline_cost = sum(gen.cost_a + gen.cost_b * gen.p_gen + gen.cost_c * (gen.p_gen ** 2) 
                       for gen in test_network.generators)
    baseline_losses = sum(branch.resistance * 1.0 for branch in test_network.branches)  # Simplified
    baseline_voltage_dev = sum((bus.voltage_mag - 1.0)**2 for bus in test_network.buses)
    
    # Add FACTS device and re-evaluate
    facts_device = FACTSDevice(device_type, location, rating, {})
    test_network.add_facts_device(facts_device)
    
    facts_solver = FACTSIntegratedPowerFlow(test_network)
    facts_solver.solve_power_flow_with_facts()
    
    facts_cost = sum(gen.cost_a + gen.cost_b * gen.p_gen + gen.cost_c * (gen.p_gen ** 2) 
                    for gen in test_network.generators)
    facts_losses = sum(branch.resistance * 1.0 for branch in test_network.branches)  # Simplified
    facts_voltage_dev = sum((bus.voltage_mag - 1.0)**2 for bus in test_network.buses)
    
    # Calculate improvements
    benefits = {
        'cost_reduction_percent': ((baseline_cost - facts_cost) / baseline_cost) * 100,
        'loss_reduction_percent': ((baseline_losses - facts_losses) / baseline_losses) * 100,
        'voltage_improvement_percent': ((baseline_voltage_dev - facts_voltage_dev) / baseline_voltage_dev) * 100,
        'total_benefit_score': 0.0
    }
    
    # Calculate composite benefit score
    benefits['total_benefit_score'] = (benefits['cost_reduction_percent'] + 
                                     benefits['loss_reduction_percent'] + 
                                     benefits['voltage_improvement_percent']) / 3.0
    
    return benefits

def generate_facts_placement_candidates(network: PowerSystemNetwork, device_type: FACTSType) -> List[Union[int, Tuple[int, int]]]:
    """Generate candidate locations for FACTS device placement"""
    candidates = []
    
    if device_type == FACTSType.SVC:
        # SVC can be placed at any load bus
        for bus in network.buses:
            if bus.bus_type == 1 and (bus.load_p > 0 or bus.load_q > 0):  # Load bus
                candidates.append(bus.number)
    
    elif device_type in [FACTSType.TCSC, FACTSType.UPFC]:
        # Line-connected devices
        for branch in network.branches:
            candidates.append((branch.from_bus, branch.to_bus))
    
    elif device_type == FACTSType.DPFC:
        # DPFC requires multiple lines - generate combinations
        lines = [(branch.from_bus, branch.to_bus) for branch in network.branches]
        # For simplicity, consider pairs of lines
        for i in range(len(lines)):
            for j in range(i + 1, min(i + 3, len(lines))):  # Limit to nearby lines
                candidates.append([lines[i], lines[j]])
    
    return candidates
'''

# Save FACTS models
with open(f'{output_dir}/facts_models.py', 'w') as f:
    f.write(facts_models_code)

print("✓ FACTS device models created")